# Coleta dados apifootball
## Etapa coleta dos dados e gravação na bronze
Esta etapa consiste em fazer uma consulta e gravar os dados em uma camada bronze, adicionando apenas um campo para identificar a data e hora da coleta.

A coleta ocorre por meio da API https://apifootball.com/.

O método usado é o get_events, documentado [aqui](https://apifootball.com/documentation/#Events).

In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import lit, col
from datetime import datetime, timedelta

In [0]:
api_key = dbutils.fs.head("dbfs:/Volumes/apifootball/bronze/arquivos/api_key_g.txt")
#api_key = (spark.read.text("/Volumes/apifootball/bronze/arquivos/apifootball_key.txt").first()[0].strip())

In [0]:
url = (
    f'https://apiv3.apifootball.com/'
    f'?action=get_events'
    f'&from=2026-01-01'
    f'&to=2026-12-31'
    f'&league_id=99' #brasil
    f'&APIkey={api_key}'
)

# Request
response = requests.get(url)

# Status
print(response.status_code)

# JSON
data = response.json()

# DataFrame
df_raw = pd.DataFrame(data)

dh_processing_br = datetime.now() - timedelta(hours=3)
df_raw = df_raw.withColumn(
    "dh_processing_br",
    lit(dh_processing_br)
)

df_raw.createOrReplaceTempView("vw_raw")
# Visualizar
print((df_raw.count(), len(df_raw.columns)))

In [0]:
spark.sql(
    """
        delete from apifootball.bronze.matches_raw as mr
        where cast(mr.dh_processing_br as date) in (select cast(d.dh_processing_br as date) from vw_raw as d)
    """
)

In [0]:
df_raw.write.mode("append").saveAsTable("apifootball.bronze.matches_raw")

In [0]:
spark.sql(
    """
    delete from apifootball.bronze.matches_raw as mr
    where year(mr.dh_processing_br) < (year(current_date)-2)
"""
)